# Feature Analysis - Epstein Accountability Index

Attribution: Scaffolded with AI assistance (Claude, Anthropic)

This notebook analyzes the extracted features and their relationships.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr

sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

%matplotlib inline

## Load Feature Matrix

In [ ]:
features_df = pd.read_csv('../data/processed/features.csv')
consequences_df = pd.read_csv('../data/processed/consequences.csv')

# Merge
df = features_df.merge(consequences_df[['name', 'consequence_tier']], on='name', how='left')

print(f"Feature matrix shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst few rows:")
df.head()

## Feature Distributions

In [ ]:
# Plot distributions of numerical features
numerical_features = [
    'mention_count', 'total_mentions', 'mean_context_sentiment',
    'cooccurrence_score', 'doc_type_diversity', 'severity_score'
]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, feature in enumerate(numerical_features):
    axes[i].hist(df[feature].dropna(), bins=30, edgecolor='black')
    axes[i].set_xlabel(feature)
    axes[i].set_ylabel('Frequency')
    axes[i].set_title(f'Distribution of {feature}')

plt.tight_layout()
plt.show()

## Feature Correlations

In [ ]:
# Compute correlation matrix
corr_features = numerical_features + ['consequence_tier']
correlation_matrix = df[corr_features].corr()

# Plot heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## Feature Importance by Consequence Tier

In [ ]:
# Analyze features by consequence tier
tier_analysis = df.groupby('consequence_tier')[numerical_features].mean()
print("Mean feature values by consequence tier:")
tier_analysis

In [ ]:
# Visualize feature differences across tiers
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, feature in enumerate(numerical_features):
    df.boxplot(column=feature, by='consequence_tier', ax=axes[i])
    axes[i].set_xlabel('Consequence Tier')
    axes[i].set_ylabel(feature)
    axes[i].set_title(f'{feature} by Consequence Tier')
    axes[i].get_figure().suptitle('')

plt.tight_layout()
plt.show()

## Statistical Tests

In [ ]:
# Pearson correlations with consequence tier
print("Correlations with consequence tier:\n")

for feature in numerical_features:
    valid_data = df[[feature, 'consequence_tier']].dropna()
    if len(valid_data) > 0:
        corr, p_value = pearsonr(valid_data[feature], valid_data['consequence_tier'])
        significance = '***' if p_value < 0.001 else '**' if p_value < 0.01 else '*' if p_value < 0.05 else ''
        print(f"{feature:25s}: r={corr:6.3f}, p={p_value:.4f} {significance}")